# Tool-Router Ladder: Stage B, teacher-forced generation for Qwen2.5-3B (V1-17)

The 3B rows of the forced-eval tables: fine-tuned and zero-shot at catalogs 16, 80 and 40, on
the same 1,416 steps as 0.5B and 1.5B. Run after `kaggle_stage_b_train` has pushed the adapter.

**Settings:** GPU T4, Internet on, the dataset attached, secret `HF_TOKEN` ticked.
**Run:** Save Version → **Save & Run All (Commit)**. **No API quota.**

Order: the fine-tuned adapter first (c16, c80, c40), then zero-shot. 3B is about twice as slow
as 1.5B, so this may need two sessions: each run is pushed to the Hub as soon as it finishes,
the next session pulls the files back, and `generate.py` resumes from them. Just rerun it.

In [ ]:
# Secrets: Add-ons -> Secrets, and TICK each one for this notebook.
import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for key in ["HF_TOKEN"]:
    os.environ[key] = _s.get_secret(key)
print("secrets loaded:", ["HF_TOKEN"])

In [ ]:
!pip install -q vllm

# Step out of the repo BEFORE deleting it: rmtree on the kernel's own working
# directory leaves cwd pointing at a dead inode, and then every ! command fails
# with "getcwd: cannot access parent directories" -- including this git clone.
%cd /kaggle/working
import shutil
shutil.rmtree("/kaggle/working/the_llm_project", ignore_errors=True)
!git clone -q https://github.com/madhusiddharths/the_llm_project.git /kaggle/working/the_llm_project
%cd /kaggle/working/the_llm_project
!git log -1 --oneline

In [ ]:
# Finds the attached dataset wherever Kaggle mounted it (it must contain manifest.json).
import glob, os
hits = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert hits, "attach the tool-router dataset (Add Input) - no manifest.json under /kaggle/input"
DATA = os.path.dirname(hits[0])
HF_USER = "madhusiddharths1"   # your Hugging Face username
print("DATA =", DATA); print(sorted(os.listdir(DATA)))

OUT = '/kaggle/working/out'
os.makedirs(OUT, exist_ok=True)

In [ ]:
# Resume across sessions: bring back any 3B completions an earlier session pushed.
import shutil
from huggingface_hub import HfApi, snapshot_download
REPO = f"{HF_USER}/tool-router-results"
api = HfApi()
api.create_repo(REPO, repo_type="dataset", private=True, exist_ok=True)
snap = snapshot_download(REPO, repo_type="dataset", allow_patterns=["completions/completions-qwen3b-*"])
for f in glob.glob(f"{snap}/completions/completions-qwen3b-*"):
    shutil.copy(f, OUT)
print("resuming from:", sorted(os.listdir(OUT)))

def push(path):
    for f in (path, path.replace(".jsonl", ".meta.json")):
        if os.path.exists(f):
            api.upload_file(path_or_fileobj=f, path_in_repo=f"completions/{os.path.basename(f)}",
                            repo_id=REPO, repo_type="dataset")

In [ ]:
# One run at a time, pushed as soon as it finishes, so a session that times out keeps its work.
runs = [("sft", c) for c in (16, 80, 40)] + [("zeroshot", c) for c in (16, 80, 40)]
for arm, catalog in runs:
    out = f"{OUT}/completions-qwen3b-{arm}-c{catalog}.jsonl"
    prompts = f"{DATA}/prompts-c{catalog}.jsonl"
    print(f"===== qwen3b {arm} c{catalog} =====")
    if arm == "sft":
        !python src/generate.py --config configs/qwen3b.yaml --adapter "$HF_USER/tool-router-qwen3b-sft" --prompts "{prompts}" --out "{out}"
    else:
        !python src/generate.py --config configs/qwen3b.yaml --prompts "{prompts}" --out "{out}"
    push(out)
print("pushed:", sorted(os.listdir(OUT)))